# Notebook 01 — DFDC Download and Stratified Actor-Disjoint Sampling

**Goal:** Build a 30,000-clip subset of DFDC with:
- Perfect class balance (15K REAL / 15K FAKE)
- Actor-disjoint train/val/test splits (24K / 3K / 3K)
- Deterministic, reproducible (seed = 42)
- Resumable (can be killed and restarted)

**Output:**
- `data/raw_clips/*.mp4` — 30K sampled clip files
- `data/manifest.csv` — clip_id, label, actor_id, split, source_part
- `data/splits/{train,val,test}.csv` — per-split manifests
- `data/processed_parts.txt` — bookkeeping for resumability

**Approach:** Greedy single-pass over DFDC parts. For each part:
1. Download the zip
2. Read metadata.json from inside the zip (no full extraction)
3. Group clips by actor (`original` field)
4. Assign each new actor deterministically to train/val/test via MD5 hash
5. Take 1 REAL + 1 FAKE from each actor whose target split isn't yet full
6. Extract only those clips from the zip
7. Delete the zip
8. Update manifests
9. Stop when all splits hit their targets

**Runtime:** ~6-10 hours depending on Kaggle bandwidth.

**Before running:**
- Verify `kaggle.json` is at `C:\Users\<username>\.kaggle\kaggle.json`
- Verify you've accepted DFDC rules at https://www.kaggle.com/c/deepfake-detection-challenge/rules
- Verify free disk space ≥ 200 GB

## 1. Imports and configuration

In [ ]:
import os
import sys
import json
import shutil
import hashlib
import zipfile
import subprocess
import shutil as _shutil
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from tqdm.auto import tqdm

print(f"Python: {sys.version.split()[0]}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ============================================================
# CONFIGURATION — adjust here, not in code below
# ============================================================

# Project root — assumes notebook is run from notebooks/ subfolder
PROJECT_ROOT = Path('..').resolve()

# Data paths
DATA_DIR        = PROJECT_ROOT / 'data'
RAW_ZIPS_DIR    = DATA_DIR / 'raw_zips'       # transient: zips being processed
RAW_CLIPS_DIR   = DATA_DIR / 'raw_clips'      # output: sampled mp4 files
SPLITS_DIR      = DATA_DIR / 'splits'
MANIFEST_CSV    = DATA_DIR / 'manifest.csv'
PROCESSED_PARTS = DATA_DIR / 'processed_parts.txt'

# Sampling targets — split sizes in CLIPS (not actors)
# Since we take 2 clips per actor (1 real + 1 fake), actor counts are half these
TARGET_PER_SPLIT = {
    'train': 24_000,
    'val':   3_000,
    'test':  3_000,
}
TARGET_TOTAL = sum(TARGET_PER_SPLIT.values())

# Split ratios used for hashing (must sum to 1.0)
SPLIT_RATIOS = {'train': 0.80, 'val': 0.10, 'test': 0.10}

# Reproducibility seed for actor → split hashing
SAMPLE_SEED = 42

# Per-actor clip counts (kept tight for max actor diversity + balance)
REALS_PER_ACTOR = 1
FAKES_PER_ACTOR = 1

# How many DFDC parts exist on Kaggle
N_DFDC_PARTS = 50  # parts 0..49

# Kaggle competition slug
KAGGLE_COMP = 'deepfake-detection-challenge'

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Target total clips: {TARGET_TOTAL:,}")
print(f"Per split:           {TARGET_PER_SPLIT}")
print(f"Clips per actor:     {REALS_PER_ACTOR} real + {FAKES_PER_ACTOR} fake = {REALS_PER_ACTOR + FAKES_PER_ACTOR}")
print(f"Estimated actors needed: {TARGET_TOTAL // (REALS_PER_ACTOR + FAKES_PER_ACTOR):,}")

## 2. Folder setup

Creates the data folders we need. Idempotent — safe to re-run.

In [ ]:
for d in [DATA_DIR, RAW_ZIPS_DIR, RAW_CLIPS_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  OK: {d}")

print("\nFolder structure ready.")

## 3. Kaggle credentials check

Verifies that `kaggle.json` is in place and the API works.

In [ ]:
# Where Kaggle CLI looks for credentials on Windows
kaggle_json_path = Path(os.environ['USERPROFILE']) / '.kaggle' / 'kaggle.json'

if not kaggle_json_path.exists():
    print(f"[ERROR] {kaggle_json_path} not found.")
    print("\nSteps to fix:")
    print("  1. Go to https://www.kaggle.com/settings")
    print("  2. Click 'Create New API Token' — downloads kaggle.json")
    print(f"  3. Move it to: {kaggle_json_path.parent}\\")
    print("  4. Accept DFDC rules: https://www.kaggle.com/c/deepfake-detection-challenge/rules")
    raise FileNotFoundError("Kaggle credentials missing")

print(f"[OK] Kaggle credentials found at {kaggle_json_path}")

# Test the API by listing competition files (only a few, to keep output short)
result = subprocess.run(
    ['kaggle', 'competitions', 'files', '-c', KAGGLE_COMP],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("[ERROR] kaggle CLI failed:")
    print(result.stderr)
    print("\nDid you accept the DFDC competition rules?")
    raise RuntimeError("Kaggle API not working")

print("[OK] Kaggle API responsive")
print("\nFirst few lines of `kaggle competitions files`:")
print('\n'.join(result.stdout.splitlines()[:8]))

## 4. Helper functions

Pure functions, no side effects. Each is unit-testable in principle.

In [ ]:
def actor_to_split(actor_id: str, ratios: dict = SPLIT_RATIOS, seed: int = SAMPLE_SEED) -> str:
    """Map an actor_id to a split deterministically via MD5 hash.

    Same actor_id always returns the same split across runs. This is the
    core of actor-disjoint splitting: every clip from a given actor lands
    in the same split, with zero leakage between train/val/test.

    Args:
        actor_id: The actor identifier (we use DFDC's `original` field).
        ratios: Dict mapping split name → fraction. Must sum to 1.0.
        seed: Salt prepended to the hash input for reproducibility.

    Returns:
        One of 'train', 'val', 'test'.
    """
    # Hash with seed as salt — different seed gives different splits
    h = hashlib.md5(f"{seed}_{actor_id}".encode()).hexdigest()
    # Convert hex digest to a float in [0, 1)
    n = int(h, 16) / (16 ** 32)
    
    cum = 0.0
    for split, ratio in ratios.items():
        cum += ratio
        if n < cum:
            return split
    return list(ratios.keys())[-1]  # numerical edge case fallback


# Quick sanity check: same input → same output, distribution roughly matches ratios
test_actors = [f"actor_{i:05d}.mp4" for i in range(10_000)]
test_splits = [actor_to_split(a) for a in test_actors]
test_counts = Counter(test_splits)
print("Hash distribution check (10k synthetic actors):")
for split in ['train', 'val', 'test']:
    pct = test_counts[split] / 10_000 * 100
    expected = SPLIT_RATIOS[split] * 100
    print(f"  {split}: {test_counts[split]:,} ({pct:.1f}% — expected {expected:.0f}%)")

# Determinism check
assert actor_to_split('test_actor_123.mp4') == actor_to_split('test_actor_123.mp4'), "Hash not deterministic!"
print("\n[OK] Hashing is deterministic and roughly balanced.")

In [ ]:
def get_free_gb(path: Path) -> float:
    """Return free disk space at path in GB."""
    return _shutil.disk_usage(path).free / (1024**3)


def download_part(part_idx: int, output_dir: Path, retries: int = 2) -> Path:
    """Download one DFDC part zip via the Kaggle CLI.

    The Kaggle CLI shows its own progress bar. We let it print to stdout.
    Retries on transient network errors.

    Returns the path to the downloaded zip.
    """
    filename = f"dfdc_train_part_{part_idx}.zip"
    zip_path = output_dir / filename
    
    if zip_path.exists():
        size_gb = zip_path.stat().st_size / (1024**3)
        print(f"  Zip already on disk ({size_gb:.1f} GB), skipping download.")
        return zip_path
    
    cmd = [
        'kaggle', 'competitions', 'download',
        '-c', KAGGLE_COMP,
        '-f', filename,
        '-p', str(output_dir),
    ]
    
    for attempt in range(retries + 1):
        result = subprocess.run(cmd, capture_output=False)
        if result.returncode == 0 and zip_path.exists():
            return zip_path
        print(f"  [Attempt {attempt+1}/{retries+1}] download failed (rc={result.returncode}), retrying...")
    
    raise RuntimeError(f"Failed to download {filename} after {retries+1} attempts")


def read_metadata_from_zip(zip_path: Path) -> dict:
    """Read metadata.json from inside a zip without extracting all of it.

    DFDC zips contain hundreds of mp4 files and one metadata.json. We only
    need the metadata.json to decide which clips to keep, so we read it
    in-place without unzipping the rest.
    """
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith('metadata.json'):
                with zf.open(name) as f:
                    return json.load(f)
    raise FileNotFoundError(f"No metadata.json in {zip_path}")


def extract_clips_from_zip(zip_path: Path, clip_names: list, output_dir: Path) -> int:
    """Extract specific .mp4 files from a DFDC zip.

    The clips inside the zip may live in a subfolder; we flatten them all
    into output_dir for simplicity.

    Returns the number of clips actually extracted (some may be missing).
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    extracted = 0
    with zipfile.ZipFile(zip_path) as zf:
        # Build basename → full_path lookup so we don't care about subfolders
        name_map = {Path(n).name: n for n in zf.namelist() if n.endswith('.mp4')}
        for clip in clip_names:
            full_path = name_map.get(clip)
            if full_path is None:
                continue  # clip listed in metadata but not in zip (shouldn't happen)
            dst = output_dir / clip
            if dst.exists():
                extracted += 1
                continue  # already extracted in a previous run
            with zf.open(full_path) as src, open(dst, 'wb') as out:
                shutil.copyfileobj(src, out)
            extracted += 1
    return extracted

In [ ]:
def sample_from_part_metadata(metadata: dict,
                              seen_actors: set,
                              current_counts: dict,
                              target_counts: dict = TARGET_PER_SPLIT) -> list:
    """Decide which clips to keep from one part's metadata.

    For each unseen actor in this part:
      1. Hash the actor_id → assigns to train/val/test
      2. If that split is already full, skip the actor
      3. Otherwise take REALS_PER_ACTOR real clips + FAKES_PER_ACTOR fake clips
      4. Mark the actor as seen so it can never appear in another part's sample

    Updates current_counts in place.

    Returns: list of dicts with keys: clip_name, label, actor_id, split.
    """
    # Group this part's clips by actor (using `original` as actor proxy)
    by_actor = defaultdict(lambda: {'REAL': [], 'FAKE': []})
    for clip_name, info in metadata.items():
        if not clip_name.endswith('.mp4'):
            continue
        label = info.get('label', '').upper()
        if label not in ('REAL', 'FAKE'):
            continue
        # For REAL clips: actor_id is the clip itself (since original == self)
        # For FAKE clips: actor_id is the source clip (`original` field)
        actor_id = info.get('original') or clip_name
        by_actor[actor_id][label].append(clip_name)
    
    selected = []
    
    for actor_id, clips in by_actor.items():
        if actor_id in seen_actors:
            continue
        
        # Need both real AND fake clips for this actor to maintain balance
        if not clips['REAL'] or not clips['FAKE']:
            continue
        
        # Decide this actor's split
        split = actor_to_split(actor_id)
        
        # If this split is already full, mark actor as seen but don't sample
        # (otherwise we'd accidentally include this actor when its split has space)
        # Edge case: if we skip a full split, we should NOT mark seen — leave it
        # so a different hash seed would still be deterministic. But for resumability,
        # we mark seen here so we don't re-process.
        if current_counts[split] + REALS_PER_ACTOR + FAKES_PER_ACTOR > target_counts[split]:
            seen_actors.add(actor_id)
            continue
        
        # Take first N reals + first N fakes (deterministic ordering = whatever order
        # Python dict gives us; safer to sort by clip_name)
        chosen_reals = sorted(clips['REAL'])[:REALS_PER_ACTOR]
        chosen_fakes = sorted(clips['FAKE'])[:FAKES_PER_ACTOR]
        
        for clip in chosen_reals:
            selected.append({
                'clip_name': clip, 'label': 'REAL', 'actor_id': actor_id, 'split': split,
            })
        for clip in chosen_fakes:
            selected.append({
                'clip_name': clip, 'label': 'FAKE', 'actor_id': actor_id, 'split': split,
            })
        
        current_counts[split] += REALS_PER_ACTOR + FAKES_PER_ACTOR
        seen_actors.add(actor_id)
    
    return selected

## 5. Resume state

Load any state from a previous incomplete run. This is what makes the notebook safely interruptible — if AnyDesk drops mid-download, we pick up from where we left off.

In [ ]:
# Load manifest if it exists (resuming a previous run)
if MANIFEST_CSV.exists():
    manifest_df = pd.read_csv(MANIFEST_CSV)
    print(f"[RESUME] Loaded existing manifest with {len(manifest_df):,} clips")
    seen_actors = set(manifest_df['actor_id'].unique())
    current_counts = manifest_df.groupby('split').size().to_dict()
    # Make sure all splits are represented in the dict
    for split in TARGET_PER_SPLIT:
        current_counts.setdefault(split, 0)
else:
    manifest_df = pd.DataFrame(columns=['clip_name', 'label', 'actor_id', 'split', 'source_part'])
    seen_actors = set()
    current_counts = {split: 0 for split in TARGET_PER_SPLIT}
    print("[FRESH START] No existing manifest, starting from scratch")

# Load list of already-processed parts
if PROCESSED_PARTS.exists():
    processed_parts = set(int(line.strip()) for line in PROCESSED_PARTS.read_text().splitlines() if line.strip())
    print(f"[RESUME] {len(processed_parts)} parts already processed: {sorted(processed_parts)}")
else:
    processed_parts = set()

print("\nCurrent state:")
for split in ['train', 'val', 'test']:
    target = TARGET_PER_SPLIT[split]
    cur = current_counts[split]
    pct = cur / target * 100
    bar = '#' * int(pct / 5) + '-' * (20 - int(pct / 5))
    print(f"  {split}: [{bar}] {cur:,}/{target:,} ({pct:.1f}%)")
print(f"  Total actors seen: {len(seen_actors):,}")

## 6. Main loop — download → sample → extract → delete

Iterates through DFDC parts in order. Stops when all splits hit their targets.

**This is the long-running cell.** Plan for ~6-10 hours. Output streams as we go.

If interrupted (AnyDesk drop, kernel kill, anything), just re-run this cell — it picks up from `processed_parts.txt`.

In [ ]:
def all_splits_full() -> bool:
    return all(current_counts[s] >= TARGET_PER_SPLIT[s] for s in TARGET_PER_SPLIT)


def save_state(new_rows: list, part_idx: int):
    """Append new rows to manifest CSV and mark part as processed."""
    global manifest_df
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        manifest_df = pd.concat([manifest_df, new_df], ignore_index=True)
        manifest_df.to_csv(MANIFEST_CSV, index=False)
    # Record this part as done
    with open(PROCESSED_PARTS, 'a') as f:
        f.write(f"{part_idx}\n")
    processed_parts.add(part_idx)


for part_idx in range(N_DFDC_PARTS):
    if all_splits_full():
        print(f"\n[DONE] All splits filled after {part_idx} parts. Stopping.")
        break
    
    if part_idx in processed_parts:
        continue
    
    print(f"\n{'='*60}")
    print(f"PART {part_idx}/{N_DFDC_PARTS - 1}")
    print(f"  Current counts: train={current_counts['train']:,}, val={current_counts['val']:,}, test={current_counts['test']:,}")
    print(f"  Free disk: {get_free_gb(DATA_DIR):.1f} GB")
    print(f"{'='*60}")
    
    # Disk space safety check — bail if dangerously low
    if get_free_gb(DATA_DIR) < 30:
        print("[WARN] Less than 30 GB free, stopping to avoid filling disk.")
        break
    
    # ---- Download zip ----
    print(f"\n[1/4] Downloading dfdc_train_part_{part_idx}.zip ...")
    try:
        zip_path = download_part(part_idx, RAW_ZIPS_DIR)
    except Exception as e:
        print(f"  [ERROR] {e}")
        print("  Skipping this part for now; will retry on next run.")
        continue
    
    # ---- Read metadata ----
    print(f"\n[2/4] Reading metadata from zip ...")
    metadata = read_metadata_from_zip(zip_path)
    print(f"  {len(metadata)} clip entries in metadata.json")
    
    # ---- Sample clips for this part ----
    print(f"\n[3/4] Sampling clips from part {part_idx} ...")
    selected = sample_from_part_metadata(metadata, seen_actors, current_counts)
    print(f"  Selected {len(selected)} clips ({len(selected)//2} new actors)")
    
    # Stamp source_part on each row before saving
    for row in selected:
        row['source_part'] = part_idx
    
    # ---- Extract selected clips and delete zip ----
    if selected:
        print(f"\n[4/4] Extracting {len(selected)} clips ...")
        clip_names_to_extract = [row['clip_name'] for row in selected]
        n_extracted = extract_clips_from_zip(zip_path, clip_names_to_extract, RAW_CLIPS_DIR)
        print(f"  Extracted {n_extracted}/{len(clip_names_to_extract)} clips")
        save_state(selected, part_idx)
    else:
        print("\n[4/4] No clips selected from this part. Marking as processed anyway.")
        save_state([], part_idx)
    
    # ---- Always delete the zip to free disk ----
    zip_path.unlink()
    print(f"  Deleted zip. Free disk now: {get_free_gb(DATA_DIR):.1f} GB")

print("\n" + "="*60)
print("DOWNLOAD + SAMPLING COMPLETE")
print("="*60)
print(f"  Total clips sampled:  {len(manifest_df):,}")
print(f"  Unique actors:        {manifest_df['actor_id'].nunique():,}")
print(f"  Parts processed:      {len(processed_parts)}")

## 7. Verification — split sizes, balance, leakage check

Critical sanity checks before we move on to preprocessing. Any failure here means our sampling logic has a bug.

In [ ]:
# Reload manifest to be sure we're looking at the disk version
manifest_df = pd.read_csv(MANIFEST_CSV)

print("=" * 60)
print("MANIFEST OVERVIEW")
print("=" * 60)
print(f"Total clips:     {len(manifest_df):,}")
print(f"Unique actors:   {manifest_df['actor_id'].nunique():,}")
print(f"Unique source parts: {manifest_df['source_part'].nunique()}")

print("\n--- Split sizes ---")
split_label_counts = manifest_df.groupby(['split', 'label']).size().unstack(fill_value=0)
split_label_counts['TOTAL'] = split_label_counts.sum(axis=1)
print(split_label_counts)

print("\n--- Class balance per split ---")
for split in ['train', 'val', 'test']:
    sub = manifest_df[manifest_df['split'] == split]
    n_real = (sub['label'] == 'REAL').sum()
    n_fake = (sub['label'] == 'FAKE').sum()
    total = len(sub)
    if total > 0:
        print(f"  {split}: REAL={n_real:,} ({n_real/total*100:.1f}%), FAKE={n_fake:,} ({n_fake/total*100:.1f}%)")

# ---- Critical leakage check ----
print("\n--- Actor-disjoint leakage check ---")
actor_split_map = manifest_df.groupby('actor_id')['split'].nunique()
leaked_actors = actor_split_map[actor_split_map > 1]
if len(leaked_actors) == 0:
    print("  [OK] Every actor appears in exactly one split. No leakage.")
else:
    print(f"  [FAIL] {len(leaked_actors)} actors appear in multiple splits!")
    print(leaked_actors.head())
    raise AssertionError("Actor leakage detected — sampling logic has a bug")

# ---- File existence check ----
print("\n--- Disk file existence check ---")
missing = []
for clip in manifest_df['clip_name']:
    if not (RAW_CLIPS_DIR / clip).exists():
        missing.append(clip)
print(f"  Manifest claims {len(manifest_df):,} clips, found {len(manifest_df) - len(missing):,} on disk")
if missing:
    print(f"  [WARN] {len(missing)} clips missing from disk — may need to re-extract")
    print(f"  First few missing: {missing[:5]}")
else:
    print("  [OK] All clips present on disk")

## 8. Write per-split CSVs for downstream notebooks

In [ ]:
for split in ['train', 'val', 'test']:
    split_df = manifest_df[manifest_df['split'] == split].copy()
    out_path = SPLITS_DIR / f'{split}.csv'
    split_df.to_csv(out_path, index=False)
    print(f"  Wrote {out_path}: {len(split_df):,} clips")

print("\nFinal manifest paths:")
print(f"  Master:  {MANIFEST_CSV}")
print(f"  Train:   {SPLITS_DIR / 'train.csv'}")
print(f"  Val:     {SPLITS_DIR / 'val.csv'}")
print(f"  Test:    {SPLITS_DIR / 'test.csv'}")

## 9. Disk usage summary

In [ ]:
def folder_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    total = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
    return total / (1024**3)

print("Disk usage:")
print(f"  data/raw_zips/   : {folder_size_gb(RAW_ZIPS_DIR):.2f} GB (should be ~0 — zips deleted as we go)")
print(f"  data/raw_clips/  : {folder_size_gb(RAW_CLIPS_DIR):.2f} GB")
print(f"  data/splits/     : {folder_size_gb(SPLITS_DIR):.4f} GB")
print(f"\n  Free disk:       : {get_free_gb(DATA_DIR):.1f} GB")

## 10. Next steps

If everything above checks out:
1. **Update `project_log.md`** — paste the final stats (clip counts per split, balance percentages, actors used) under Section 1 (Dataset).
2. **Move on to notebook 02** (`02_preprocessing.ipynb`) which will:
   - Run MTCNN face detection on all 30K clips
   - Extract 16-frame face crop sequences at 112×112
   - Extract audio at 16kHz, normalize to 10 seconds
   - Save preprocessed tensors for fast training I/O
   - Delete `data/raw_clips/` to recover ~120 GB

If something is off (split imbalanced, leakage detected, missing files), don't proceed — debug here first.